# STEP-048 — Ablation training session (thin caller)

Trains the four ablations of EXP0001 one after another (EXP0003 seed, EXP0004 no augmentation, EXP0005 no pretraining, EXP0006 image size 160). All logic lives in `17_Automation/` (`training`, `kaggle_sync`); every parameter lives in each `06_Experiments/EXP000N/config.yaml` (generated from `06_Experiments/ablations/STEP048_plan.yaml`). Runbook: `06_Experiments/ablations/KAGGLE_RUNBOOK.md`.

**Notebook settings:** Accelerator `GPU T4 x2`, Internet `On`, input `isthisdeception/ds0001-pp0001`. Secrets: `GITHUB_TOKEN`, `KAGGLE_USERNAME`, `KAGGLE_KEY`.

Run 0 Setup once, then A. After each experiment, A pushes its results to GitHub and then publishes its checkpoints. About 2.5–3 hours in total.

## 0. Setup

In [ ]:
import os, subprocess, sys
from datetime import datetime
from pathlib import Path

REPO_URL = 'https://github.com/isthisdeception/thesis.git'
BRANCH = 'main'
REPO = Path('/kaggle/working/thesis')
AUTOMATION = REPO / '17_Automation'
PUSH_PLAN = 'kaggle_sync/sync_plan.ablations.yaml'
RUN = ['EXP0003', 'EXP0004', 'EXP0005', 'EXP0006']   # remove finished ones when resuming in a new session

if not (REPO / '.git').exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO)], check=True)
print(subprocess.run(['git', '-C', str(REPO), 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)

In [ ]:
# Secrets (Add-ons -> Secrets, ticked for this notebook): GITHUB_TOKEN for the push, KAGGLE_* for publishing.
try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
    for _name in ('GITHUB_TOKEN', 'KAGGLE_USERNAME', 'KAGGLE_KEY'):
        try:
            os.environ[_name] = _secrets.get_secret(_name)
            print(_name, 'loaded')
        except Exception:
            print(_name, 'NOT SET')
except ImportError:
    print('kaggle_secrets unavailable (not on Kaggle)')

In [ ]:
def sh(cmd):
    """Run a shell command with live output; stop the notebook if it fails."""
    get_ipython().system(cmd)
    code = get_ipython().user_ns.get('_exit_code', 0)
    if code:
        raise RuntimeError(f'command failed with exit code {code}: {cmd}')

# Pull latest main, install environment/kaggle-requirements.txt, verify ds0001-pp0001 is attached.
sh(f'cd {AUTOMATION} && python -m kaggle_sync --plan {PUSH_PLAN} bootstrap')

In [ ]:
import torch, fastai
from importlib.metadata import version
print('python', sys.version.split()[0])
print('torch', torch.__version__, 'cuda', torch.version.cuda, 'available', torch.cuda.is_available())
print('fastai', fastai.__version__, '| plum-dispatch', version('plum-dispatch'))
print('gpu', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
assert torch.cuda.is_available(), 'GPU not visible: set Accelerator to GPU T4 x2'
for exp in RUN:
    sh(f'cd {AUTOMATION} && python -m config_validation ../06_Experiments/{exp}/config.yaml')
sh(f'cd {AUTOMATION} && python -m ablation list')

## A. Train, push, publish — one experiment at a time

Each experiment trains in its own process (about 35–40 min), then its logs, metrics, and predictions are pushed to one `kaggle/ablations-<time>` branch, then its checkpoints are published to its own Kaggle dataset. Finished experiments are skipped if this cell is run again. The first publish of each dataset prints `403 … trying to create`; that is expected.

In [ ]:
PUSH_BRANCH = f"kaggle/ablations-{datetime.now():%Y%m%d-%H%M}"
print('results branch:', PUSH_BRANCH)
for exp in RUN:
    if (REPO / '06_Experiments' / exp / 'metrics' / 'run_summary.json').exists():
        print(f'== {exp}: already trained in this session, skipping')
        continue
    print(f'== {exp}: training ({datetime.now():%H:%M})')
    sh(f'cd {AUTOMATION} && python -m training train --config ../06_Experiments/{exp}/config.yaml')
    print(f'== {exp}: pushing results')
    sh(f'cd {AUTOMATION} && python -m kaggle_sync --plan {PUSH_PLAN} push-metadata --branch {PUSH_BRANCH} --commit-message "sync: {exp} ablation results from Kaggle"')
    print(f'== {exp}: publishing checkpoints')
    sh(f'cd {AUTOMATION} && python -m kaggle_sync --plan kaggle_sync/sync_plan.{exp.lower()}.yaml publish-artifacts')
print('ALL DONE. Results branch:', PUSH_BRANCH)